<a href="https://colab.research.google.com/github/Tejaswimadastu/Generative_AI/blob/main/RAG_using_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain
!pip install -q langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -q langchain-core
!pip install -q langchain-ollama
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 23.4 MB/s eta 0:00:00


In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama

In [6]:
loader = PyPDFLoader("employee_policy.pdf")
documents = loader.load()

print("Total pages", len(documents))
print(documents[0].page_content[:1000])

Total pages 1
Employee Policy
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.


## Chunking

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))
print(chunks[0].page_content)

Total Chunks: 2
Employee Policy
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.


## Embedding

In [8]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

/tmp/ipykernel_788/2352557002.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


## Create FAISS Vector Store

In [9]:
vector_store = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS Create Successfully")

FAISS Create Successfully


## Create Retriever

In [10]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever Ready")

Retriever Ready


## Test Retrieval

In [11]:
query = "How Many casual leaves are allowed?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n Chunk {i+1}")
    print("-" * 50)
    print(doc.page_content)


 Chunk 1
--------------------------------------------------
Employee Policy
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.

 Chunk 2
--------------------------------------------------
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.


## Connect Ollama

In [13]:
llm = Ollama(model='llama3')

print("Connected to Ollama")

Connected to Ollama


/tmp/ipykernel_788/2467047543.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model='llama3')


## Build Context

In [14]:
context = "\n".join(
    [doc.page_content for doc in docs]
)

print(context)

Employee Policy
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
